In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Connect to the database using credentials from the .env file
load_dotenv("../.env")
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DATABASE_URL)

# Define the mapping for quality-related categorical features
complex_mapping = {
    'exter_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'exter_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'bsmt_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'bsmt_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'heating_qc': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'kitchen_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'fireplace_qu': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'garage_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'garage_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'pool_qc': {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'None': 0},
    'lot_shape': {'Reg': 4, 'IR1': 3, 'IR2': 2, 'IR3': 1},
    'land_slope': {'Gtl': 3, 'Mod': 2, 'Sev': 1},
    'bsmt_exposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0},
    'garage_finish': {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0},
    'utilities': {'AllPub': 4, 'NoSewr': 3, 'NoSeWa': 2, 'ELO': 1},
    'functional': {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}
}

def ordinal_encode(df, mapping):
    """Convert categorical text values into numerical scores based on the mapping."""
    df = df.copy()
    for col, m in mapping.items():
        if col in df.columns:
            df[col] = df[col].map(m).fillna(0).astype(int)
    return df

# Set the path for the best model found during the benchmarking phase
model_path = '../models/ames_best_model_catboost.pkl' 
if not os.path.exists(model_path):
    model_path = '../models/ames_xgb_model.pkl'

# Load the saved model and fetch the test set from the database
model = joblib.load(model_path)
X_test = pd.read_sql("SELECT * FROM X_test", engine)
y_test = pd.read_sql("SELECT * FROM y_test", engine).values.ravel()

# Transform features and generate predictions
X_test_encoded = ordinal_encode(X_test, complex_mapping)
log_predictions = model.predict(X_test_encoded)

# Convert log-transformed values back to actual market prices
real_predictions = np.expm1(log_predictions)
real_y_test = np.expm1(y_test)

# Calculate performance metrics to evaluate model accuracy
r2 = r2_score(real_y_test, real_predictions)
mae = mean_absolute_error(real_y_test, real_predictions)
rmse = np.sqrt(mean_squared_error(real_y_test, real_predictions))

print(f"Model File: {os.path.basename(model_path)}")
print(f"R2 Score: {r2:.4f}")
print(f"Mean Absolute Error: ${mae:.2f}")
print(f"Root Mean Squared Error: ${rmse:.2f}")